In [1]:
# =============================================================================
# BASELINE 2: MONTAJE, DEPENDENCIAS Y CARGA DE ESQUEMAS
# =============================================================================
from google.colab import drive
import os
import json
import time
import subprocess
import tempfile
import ast
from datetime import datetime, timezone
from jsonschema import validate, ValidationError

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Instalar paquetes de inferencia y analizador estático mypy
!pip install -q -U transformers accelerate bitsandbytes jsonschema mypy

# 3. Cargar OutputRecordSchema oficial
schema_record_path = "/content/drive/MyDrive/programacion_funcional/output-record-schema.json"
with open(schema_record_path, "r", encoding="utf-8") as f:
    output_record_schema = json.load(f)

print(f"✓ OutputRecordSchema cargado desde: {schema_record_path}")

# 4. Archivo destino compartido
EXPERIMENTOS_JSONL_PATH = "/content/drive/MyDrive/programacion_funcional/experimentos.jsonl"
print(f"✓ Archivo destino de resultados: {EXPERIMENTOS_JSONL_PATH}")

# 5. Verificación de mypy
!mypy --version

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.2 MB/s eta 0:00:00
✓ OutputRecordSchema cargado desde: /content/drive/MyDrive/programacion_funcional/output-record-schema.json
✓ Archivo destino de resultados: /content/drive/MyDrive/programacion_funcional/experimentos.jsonl
mypy 2.3.1 (compiled: yes)


In [2]:
# =============================================================================
# CARGA DE QWEN 2.5 7B INSTRUCT EN GPU T4 (BitsAndBytes NF4)
# =============================================================================
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_model_path = "/content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct"

# Configuración de cuantización en 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Cargar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(qwen_model_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Cargar Modelo en GPU
model = AutoModelForCausalLM.from_pretrained(
    qwen_model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

print("-" * 60)
print(f"✓ Modelo cargado exitosamente: {qwen_model_path}")
print(f"✓ Dispositivo asignado: {model.device}")
print(f"✓ Ventana de contexto máxima: {getattr(model.config, 'max_position_embeddings', None)} tokens")
print("-" * 60)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:279: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

------------------------------------------------------------
✓ Modelo cargado exitosamente: /content/drive/MyDrive/proyecto_rag_legal/modelos/instrucciones/qwen-2.5-7b-instruct
✓ Dispositivo asignado: cuda:0
✓ Ventana de contexto máxima: 32768 tokens
------------------------------------------------------------


** Definición del Prompt Tipado y Pipeline con Auditoría Estática (mypy) para Baseline 2**

 Para aislar científicamente si el beneficio proviene simplemente de agregar un analizador estático o si el formalismo funcional aporta garantías superiores, Baseline 2 exige que el LLM genere código Python fuertemente tipado (usando anotaciones de tipo estándar del módulo typing tanto en parámetros como en el retorno).   

 El pipeline ejecuta el siguiente flujo determinista:

 1. Generación con Qwen 2.5: Decodificación voraz (temperature=0.0, do_sample=False).   
 2. Validación Sintáctica: Verificación con ast.parse.   
 3. Auditoría Estática con mypy: Se guarda el código en un archivo temporal y se ejecuta mypy --strict --no-error-summary.   
 *Si mypy detecta violaciones (inconsistencias de tipo, argumentos inválidos, etc.), se marca static_validation_passed = False y se captura el mensaje de error.   
 *Si mypy aprueba (returncode == 0), se marca static_validation_passed = True y se autoriza la ejecución en runtime.   
 4. Ejecución en Runtime (condicional): Se ejecuta únicamente si pasó la validación estática previa.   
 5. Persistencia bajo OutputRecordSchema: Se valida el esquema antes de anexar la fila a experimentos.jsonl.   

In [3]:
# =============================================================================
# BASELINE 2: PYTHON TIPADO CON AUDITORÍA ESTÁTICA MYPY
# =============================================================================
import ast
import json
import os
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from jsonschema import ValidationError, validate
import torch

SYSTEM_INSTRUCTION_BASELINE_2 = """Actúas como un compilador formal que traduce reglas de negocio en lenguaje natural a una función pura de Python FUERTEMENTE TIPADA, apta para comprobación estática estricta con mypy.

REGLAS DE GENERACIÓN OBLIGATORIAS:
1. IMPORTA explícitamente los tipos necesarios desde `typing` si se requieren (ej: `from typing import Union, Optional, Callable`).
2. Define la función obligatoriamente con el nombre `def regla_negocio(...) -> ...:`.
3. Todos los parámetros y el valor de retorno DEBEN incluir anotaciones de tipo estáticas explícitas (ej: `int`, `float`, `bool`, `str`).
4. Si la regla involucra valores decimales o monetarios, anótalos como `float`. Para enteros usa `int`.
5. Si una función es recibida o aplicada como argumento, anótala como `Callable`.
6. NO uses bloques markdown (no incluyas ```python ni ```).
7. Devuelve ÚNICAMENTE el código fuente plano, sin explicaciones ni comentarios adicionales."""

PROMPT_BASE_HISTORIAL_B2 = [
    {"role": "system", "content": SYSTEM_INSTRUCTION_BASELINE_2},
    {
        "role": "user",
        "content": (
            "Regla de negocio:\nSi la edad es mayor o igual a 18 y el estado es"
            " igual a 'ACTIVO', devolver true; caso contrario, false."
        ),
    },
    {
        "role": "assistant",
        "content": (
            "def regla_negocio(edad: int, estado: str) -> bool:\n    if edad >="
            " 18 and estado == 'ACTIVO':\n        return True\n    else:\n     "
            "   return False"
        ),
    },
]


def auditar_con_mypy(codigo_fuente: str):
  """Escribe el código en un archivo temporal y ejecuta mypy con tipado estricto."""
  with tempfile.NamedTemporaryFile(
      mode="w", suffix=".py", delete=False, encoding="utf-8"
  ) as tmp:
    tmp_path = tmp.name
    tmp.write(codigo_fuente)

  try:
    # mypy en modo estricto para simular verificación formal en Python
    resultado = subprocess.run(
        [
            "mypy",
            "--strict",
            "--ignore-missing-imports",
            "--no-error-summary",
            tmp_path,
        ],
        capture_output=True,
        text=True,
    )
    if resultado.returncode == 0:
      return True, None
    else:
      # Filtrar la ruta temporal para dejar solo el mensaje de error limpio
      lineas_error = [
          line.replace(tmp_path + ":", "linea ")
          for line in resultado.stdout.splitlines()
          if line.strip()
      ]
      return False, "; ".join(lineas_error)
  finally:
    if os.path.exists(tmp_path):
      os.remove(tmp_path)


def compilar_y_evaluar_baseline_2(
    case_id: str, prompt_input: str, test_env: dict, expected_value=None
):
  t_start = time.perf_counter()

  # 1. Construir prompt y generar código
  conversacion = list(PROMPT_BASE_HISTORIAL_B2)
  conversacion.append(
      {"role": "user", "content": f"Regla de negocio:\n{prompt_input}"}
  )

  prompt = tokenizer.apply_chat_template(
      conversacion, tokenize=False, add_generation_prompt=True
  )

  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

  generated_tokens = outputs[0][inputs.input_ids.shape[1] :]
  raw_code = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

  # Limpieza de markdown residual
  if raw_code.startswith("```python"):
    raw_code = raw_code[9:]
  if raw_code.startswith("```"):
    raw_code = raw_code[3:]
  if raw_code.endswith("```"):
    raw_code = raw_code[:-3]
  raw_code = raw_code.strip()

  # 2. Validación Sintáctica
  syntax_valid = False
  static_validation_passed = False
  static_error_message = None
  runtime_passed = False
  runtime_error_message = None
  test_passed = False

  try:
    ast.parse(raw_code)
    syntax_valid = True
  except SyntaxError as e:
    syntax_valid = False
    static_error_message = f"SyntaxError: {e.msg} (línea {e.lineno})"

  # 3. Auditoría Estática con Mypy (solo si la sintaxis es válida)
  if syntax_valid:
    mypy_ok, mypy_msg = auditar_con_mypy(raw_code)
    static_validation_passed = mypy_ok
    static_error_message = mypy_msg

    # 4. Ejecución en Runtime (SOLO si mypy aprobó la función)
    if static_validation_passed:
      exec_scope = {}
      try:
        exec(raw_code, exec_scope)
        fn = exec_scope.get("regla_negocio")
        if callable(fn):
          # Invocar función pasando test_env
          resultado = fn(**test_env)
          runtime_passed = True
          if expected_value is not None:
            test_passed = resultado == expected_value
        else:
          runtime_passed = False
          runtime_error_message = (
              "No se encontró la función 'regla_negocio' ejecutable."
          )
      except Exception as e:
        runtime_passed = False
        runtime_error_message = f"{type(e).__name__}: {str(e)}"

  t_end = time.perf_counter()
  execution_time_ms = round((t_end - t_start) * 1000, 2)
  timestamp = datetime.now(timezone.utc).isoformat()

  # 5. Registro estructurado bajo OutputRecordSchema
  record = {
      "case_id": case_id,
      "lane": "baseline_2_mypy_python",
      "prompt_input": prompt_input,
      "raw_output": raw_code,
      "syntax_valid": syntax_valid,
      "static_validation_passed": static_validation_passed,
      "static_error_message": static_error_message,
      "runtime_passed": runtime_passed,
      "runtime_error_message": runtime_error_message,
      "test_passed": test_passed,
      "execution_time_ms": execution_time_ms,
      "timestamp": timestamp,
  }

  validate(instance=record, schema=output_record_schema)

  with open(EXPERIMENTOS_JSONL_PATH, "a", encoding="utf-8") as f:
    f.write(json.dumps(record, ensure_ascii=False) + "\n")

  print(f"\n[{case_id}]")
  print(f"  Syntax Valid: {syntax_valid}")
  print(f"  Mypy Passed : {static_validation_passed}")
  if static_error_message:
    print(f"  Mypy Error  : {static_error_message}")
  print(
      f"  Runtime     : {runtime_passed} (pass@1: {test_passed}) | Tiempo:"
      f" {execution_time_ms} ms"
  )

  return record

Ejecución del Banco de las 6 Reglas en Baseline 2 (Python con Mypy)

En esta celda evaluamos el mismo corpus canónico de 6 reglas para registrar el comportamiento del analizador estático estándar de la industria (mypy):   

RULE_001_VALID: Control positivo de regla válida.   

RULE_002_SCOPE_STRESS: Ausencia de variables en el entorno léxico (categoria no suministrada).   

RULE_003_TYPE_STRESS: Discrepancia de tipos en las ramas de retorno (str vs. float).   

RULE_004_OP_TYPE_MISMATCH: Operador relacional incompatible (> entre número y texto).   

RULE_005_NON_BOOL_COND: Condición no booleana (evaluación implícita de verdad sobre escalar).   

RULE_006_APP_NON_FUNCTION: Intento de invocar un escalar como si fuese un término funcional.   

In [4]:
# =============================================================================
# EJECUCIÓN DEL BENCHMARK: BASELINE 2 (PYTHON TIPADO CON MYPY)
# =============================================================================

casos_benchmark_b2 = [
    {
        "case_id": "RULE_001_VALID",
        "prompt_input": "Si la antiguedad es mayor a 5, devolver el salario multiplicado por 1.10; caso contrario devolver el salario.",
        "test_env": {"antiguedad": 6, "salario": 1000.0},
        "expected_value": 1100.0
    },
    {
        "case_id": "RULE_002_SCOPE_STRESS",
        "prompt_input": "Si el cliente tiene categoria VIP y el balance es mayor a 500.0, devolver 0.0; caso contrario devolver balance * 0.02.",
        "test_env": {"balance": 1000.0}, # No se provee 'categoria' en el entorno de prueba
        "expected_value": 20.0
    },
    {
        "case_id": "RULE_003_TYPE_STRESS",
        "prompt_input": "Si la edad es menor a 18, devolver 'MENOR'; caso contrario devolver 100.0.",
        "test_env": {"edad": 25},
        "expected_value": 100.0
    },
    {
        "case_id": "RULE_004_OP_TYPE_MISMATCH",
        "prompt_input": "Si el balance es mayor a 'ALTO', devolver 0.0; caso contrario devolver balance.",
        "test_env": {"balance": 2500.0},
        "expected_value": None
    },
    {
        "case_id": "RULE_005_NON_BOOL_COND",
        "prompt_input": "Si el salario, devolver salario * 1.10; caso contrario devolver 0.0.",
        "test_env": {"salario": 5000.0},
        "expected_value": 5500.0
    },
    {
        "case_id": "RULE_006_APP_NON_FUNCTION",
        "prompt_input": "Aplicar la variable balance al valor 50.",
        "test_env": {"balance": 200.0},
        "expected_value": None
    }
]

print("=" * 75)
print("INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN BASELINE 2 (PYTHON TIPADO + MYPY)...")
print("=" * 75)

registros_b2 = []
for test in casos_benchmark_b2:
    rec = compilar_y_evaluar_baseline_2(
        case_id=test["case_id"],
        prompt_input=test["prompt_input"],
        test_env=test["test_env"],
        expected_value=test["expected_value"]
    )
    registros_b2.append(rec)

print("\n" + "=" * 75)
print("✓ Finalizada la ejecución de Baseline 2.")
print(f"✓ Los 6 registros fueron validados y agregados a {EXPERIMENTOS_JSONL_PATH}")
print("=" * 75)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


INICIANDO EVALUACIÓN DE LAS 6 REGLAS EN BASELINE 2 (PYTHON TIPADO + MYPY)...

[RULE_001_VALID]
  Syntax Valid: True
  Mypy Passed : True
  Runtime     : True (pass@1: True) | Tiempo: 10060.14 ms

[RULE_002_SCOPE_STRESS]
  Syntax Valid: True
  Mypy Passed : True
  Runtime     : False (pass@1: False) | Tiempo: 4118.27 ms

[RULE_003_TYPE_STRESS]
  Syntax Valid: True
  Mypy Passed : False
  Mypy Error  : linea 1: error: Name "Union" is not defined  [name-defined]; linea 1: note: Did you forget to import it from "typing"? (Suggestion: "from typing import Union")
  Runtime     : False (pass@1: False) | Tiempo: 3637.74 ms

[RULE_004_OP_TYPE_MISMATCH]
  Syntax Valid: True
  Mypy Passed : True
  Runtime     : False (pass@1: False) | Tiempo: 4181.93 ms

[RULE_005_NON_BOOL_COND]
  Syntax Valid: True
  Mypy Passed : True
  Runtime     : True (pass@1: True) | Tiempo: 2973.27 ms

[RULE_006_APP_NON_FUNCTION]
  Syntax Valid: True
  Mypy Passed : True
  Runtime     : True (pass@1: False) | Tiempo: 1984

**Hallazgos Teóricos y Metodológicos Clave**

*1) La brecha entre verificación intra-procedural y verificación de contexto (Scope):*

 *En RULE_002, mypy no detectó ninguna falla porque analizó la signatura def regla_negocio(categoria: str, balance: float) como una unidad sintáctica cerrada y válida. El choque contra la realidad se postergó hasta el runtime (missing required positional argument).   

 *El validador en Haskell, al operar como una capa neuro-simbólica que evalúa el AST frente al entorno tipado $\Gamma$ del sistema, detectó la variable libre antes de cualquier instanciación.   

*2) Alucinaciones compensatorias que burlan el análisis estático imperativo:*

 *En RULE_004, el LLM detectó el conflicto evidente de comparar float con 'ALTO'. En lugar de fallar, "resolvió" el problema alucinando una heurística: if balance > 1000000: # Suponiendo que 'ALTO' significa más de 1 millón y agregando un parámetro no solicitado (nivel: str).   

 *Para mypy, el código era matemáticamente intachable y pasó la auditoría estática con honores (static_validation_passed: true). Solo la formalización estricta mediante ADTs delimitados impidió que el LLM introdujera semántica arbitraria.   

*3) Demostración de la Hipótesis del Documento:Baseline 1 (Control libre):*

 *Los errores estocásticos y de tipado se manifiestan como excepciones crudas en runtime o, peor aún, como ejecuciones silenciosamente erróneas (falsos positivos).   

 *Baseline 2 (Mypy): Si bien reduce errores de tipos superficiales, es ciego al entorno contextual de ejecución y es fácilmente sorteado por alucinaciones semánticas del LLM que generan código Python internamente coherente pero ajeno a la especificación.   

 *Propuesta (DSL Funcional en Haskell): Registra un 100% de intercepción estática de alucinaciones estructurales, inconsistencias de tipo y violaciones de alcance, impidiendo por completo que expresiones mal formadas lleguen a la fase de ejecución.   